In [24]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [25]:
file_name = r"C:\Users\admin\Downloads\06.04.2023 £86.31 Soft99.pdf"

r"C:\Users\admin\Downloads\20.02.2023 £3,285.15 On-Line Auto Sport Limited.pdf"

'C:\\Users\\admin\\Downloads\\20.02.2023 £3,285.15 On-Line Auto Sport Limited.pdf'

In [26]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\14.03.2024 £377.14 Soft99.pdf"

In [27]:
name = "Soft99"

table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(107,340,251,453),
                  columns=[453],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

docnum = heading[0][3]
print(docnum)

date = heading[0][1]
date = str(datetime.strptime(date, "%d %b %Y"))
print(date)

ordernum = None

transfernum = None
print(ordernum)
print(transfernum)


,0
0,Invoice Date
1,14 Mar 2024
2,Invoice Number
3,INV-0620
4,Reference
5,PO34828


INV-0620
2024-03-14 00:00:00
None
None


In [28]:
with open(input_file,'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    num_pages = len(pdf_reader.pages)

print(num_pages)

2


In [29]:
#create loop to go through the pages
all_content = []
for page in range(1, num_pages + 1):
    table2 = read_pdf(input_file,
                    pages=page,
                    silent=True,
                    guess=False,
                    area=(300,9,775,573),
                    columns=[82,311,387,458,508,573],
                    pandas_options={'header': None},
                    encoding="windows-1254")
    contenti = table2[0]
    all_content.append(contenti)

content = pd.concat(all_content).reset_index(drop=True)
display(content)

,0,1,2,3,4,5
0,Item,Description,Quantity,Unit Price,VAT,Amount GBP
1,LM 00468,Fukupika Wash & Wax Wipes,10.00,6.69,20%,66.90
2,LM 10503,QJUTSU Body Coat Pro,1.00,82.12,20%,82.12
3,LM 10505,QJUTSU Wheel Coat,1.00,39.54,20%,39.54
4,LM 04111,Glaco de Cleaner,1.00,5.35,20%,5.35
5,LM,Interior Brush - black 24mm,5.00,1.82,20%,9.10
6,PWEWCZA,NaN,NaN,NaN,NaN,NaN
7,RNY24,NaN,NaN,NaN,NaN,NaN
8,LM 10309,Glaco Mirror Coat Zero,4.00,7.29,20%,29.16
9,LM 10503,QJUTSU Body Coat Pro,1.00,82.12,20%,82.12


In [30]:
# appending SKU more than 1 line
# Iterate through the DataFrame and update Column0 based on Column2
content[0] = content[0].astype(str)
for i in range(1, len(content)):
    if pd.isna(content.at[i, 2]):
        content.at[i-1,0] = content.at[i - 1, 0] + content.at[i, 0]

        if i < len(content) -1 and pd.isna(content.at[i+1, 2]):
            content.at[i-1,0] = content.at[i-1, 0] + content.at[i+1, 0]

display(content)
# Optionally remove rows where Column2 is NaN
content = content.dropna(subset=[2]).reset_index(drop=True)

display(content)


,0,1,2,3,4,5
0,Item,Description,Quantity,Unit Price,VAT,Amount GBP
1,LM 00468,Fukupika Wash & Wax Wipes,10.00,6.69,20%,66.90
2,LM 10503,QJUTSU Body Coat Pro,1.00,82.12,20%,82.12
3,LM 10505,QJUTSU Wheel Coat,1.00,39.54,20%,39.54
4,LM 04111,Glaco de Cleaner,1.00,5.35,20%,5.35
5,LMPWEWCZARNY24,Interior Brush - black 24mm,5.00,1.82,20%,9.10
6,PWEWCZARNY24,NaN,NaN,NaN,NaN,NaN
7,RNY24,NaN,NaN,NaN,NaN,NaN
8,LM 10309,Glaco Mirror Coat Zero,4.00,7.29,20%,29.16
9,LM 10503nannan,QJUTSU Body Coat Pro,1.00,82.12,20%,82.12


,0,1,2,3,4,5
0,Item,Description,Quantity,Unit Price,VAT,Amount GBP
1,LM 00468,Fukupika Wash & Wax Wipes,10.00,6.69,20%,66.90
2,LM 10503,QJUTSU Body Coat Pro,1.00,82.12,20%,82.12
3,LM 10505,QJUTSU Wheel Coat,1.00,39.54,20%,39.54
4,LM 04111,Glaco de Cleaner,1.00,5.35,20%,5.35
5,LMPWEWCZARNY24,Interior Brush - black 24mm,5.00,1.82,20%,9.10
6,LM 10309,Glaco Mirror Coat Zero,4.00,7.29,20%,29.16
7,LM 10503nannan,QJUTSU Body Coat Pro,1.00,82.12,20%,82.12
8,Payment canBank :PayrNeBank Account,be made by BACS to our Bank account. Please in...,voice number on al,l payments.,NaN,NaN
9,PAYM,ENT ADVICE,CustomerInvoice Num,ML PERber INV-062,FORMANCE0,LIMITED


In [31]:
content[2] = pd.to_numeric(content[2], errors='coerce')  # replace NaN anything that not number in column 2
content = content.dropna(subset=[2,5]).reset_index(drop=True)
content[0] = content[0].str.replace('nan','',regex=False) # remove appending nan
display(content)

,0,1,2,3,4,5
0,LM 00468,Fukupika Wash & Wax Wipes,10.0,6.69,20%,66.90
1,LM 10503,QJUTSU Body Coat Pro,1.0,82.12,20%,82.12
2,LM 10505,QJUTSU Wheel Coat,1.0,39.54,20%,39.54
3,LM 04111,Glaco de Cleaner,1.0,5.35,20%,5.35
4,LMPWEWCZARNY24,Interior Brush - black 24mm,5.0,1.82,20%,9.10
5,LM 10309,Glaco Mirror Coat Zero,4.0,7.29,20%,29.16
6,LM 10503,QJUTSU Body Coat Pro,1.0,82.12,20%,82.12


In [32]:
content.rename(columns={
    0: 'Item',
    1: 'Description',
    2: 'Quantity',
    3: 'Unit Price',
    4: 'VAT',
    5: 'Amount GBP'}, inplace=True)

display(content)

,Item,Description,Quantity,Unit Price,VAT,Amount GBP
0,LM 00468,Fukupika Wash & Wax Wipes,10.0,6.69,20%,66.90
1,LM 10503,QJUTSU Body Coat Pro,1.0,82.12,20%,82.12
2,LM 10505,QJUTSU Wheel Coat,1.0,39.54,20%,39.54
3,LM 04111,Glaco de Cleaner,1.0,5.35,20%,5.35
4,LMPWEWCZARNY24,Interior Brush - black 24mm,5.0,1.82,20%,9.10
5,LM 10309,Glaco Mirror Coat Zero,4.0,7.29,20%,29.16
6,LM 10503,QJUTSU Body Coat Pro,1.0,82.12,20%,82.12


In [33]:
dict_content = content.to_dict(orient='records')
dict_content


line_items=[]
for item in dict_content:
    # print(item)
    
    partNum = item['Item']
    desc = item['Description']
    quantity = item['Quantity']
    netTotal = item['Amount GBP']

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": partNum,
                        "name": desc,
                        "quantity": int(quantity),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

LM 00468
LM 10503
LM 10505
LM 04111
LMPWEWCZARNY24
LM 10309
LM 10503
[{'line_type': 'inventory', 'sku': 'LM 00468', 'name': 'Fukupika Wash & Wax Wipes', 'quantity': 10, 'net_total': 66.9, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LM 10503', 'name': 'QJUTSU Body Coat Pro', 'quantity': 1, 'net_total': 82.12, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LM 10505', 'name': 'QJUTSU Wheel Coat', 'quantity': 1, 'net_total': 39.54, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LM 04111', 'name': 'Glaco de Cleaner', 'quantity': 1, 'net_total': 5.35, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LMPWEWCZARNY24', 'name': 'Interior Brush - black 24mm', 'quantity': 5, 'net_total': 9.1, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LM 10309', 'name': 'Glaco Mirror Coat Zero', 'quantity': 4, 'net_total': 29.16, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'LM 10503', 'name': 'QJUTSU Body Coat Pro', 'quantity': 1, 'net_total'

In [34]:
table4 = read_pdf(input_file,
            pages="1",
            silent=True,
            guess=False,
            area=(364,400,794,575),
            columns=[516,575],
            pandas_options={'header': None},
            encoding='windows-1254')

total_content=table4[0]

#To remove "£"
total_content[[1]] = total_content[[1]].replace('[£, ]','', regex=True).astype('string')
#print(total_content)

total_content = total_content.dropna(subset=[0,1])  # Remove rows with NaN in column 0
total_content = total_content.dropna(subset=[0]).reset_index(drop=True)
total_content


,0,1
0,39.54 20%,39.54
1,5.35 20%,5.35
2,1.82 20%,9.1
3,7.29 20%,29.16
4,82.12 20%,82.12
5,Subtotal,314.29
6,TOTAL VAT 20%,62.85
7,TOTAL GBP,377.14


In [35]:
row_index = total_content.index[total_content[0] == "TOTAL GBP"].tolist()[0]


final_total = str(total_content[1][row_index])
final_total = float(final_total.replace(","," "))
final_total

377.14

In [36]:
## merge invoice heading and line items information..
## ..into one dict variable, 'payload'
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        docnum,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\06.04.2023 £86.31 Soft99.pdf',
 'Type': 'Products',
 'Name': 'Soft99',
 'Date': '2024-03-14 00:00:00',
 'Reference No.': 'INV-0620',
 'Order No.': None,
 'Transfer No.': None,
 'Document No.': 'INV-0620',
 'Line Items': [{'line_type': 'inventory',
   'sku': 'LM 00468',
   'name': 'Fukupika Wash & Wax Wipes',
   'quantity': 10,
   'net_total': 66.9,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'LM 10503',
   'name': 'QJUTSU Body Coat Pro',
   'quantity': 1,
   'net_total': 82.12,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'LM 10505',
   'name': 'QJUTSU Wheel Coat',
   'quantity': 1,
   'net_total': 39.54,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'LM 04111',
   'name': 'Glaco de Cleaner',
   'quantity': 1,
   'net_total': 5.35,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'LMPWEWCZARNY24',
   'name': 'Interior Brush - black 24mm',
   'quantity': 5,
   'net_t